# Phi-4-mini Socratic tutor: baseline, LoRA tuning and final fine-tune (Kaggle T4)

Model: `microsoft/Phi-4-mini-instruct` @ `cfbefacb` (MIT), loaded in 4-bit NF4.

This notebook is the Qwen 3.5-4B notebook by Hariom Kumar (`qwen-finetuning.ipynb`) with only the model-specific
parts changed, so Phi-4-mini can be compared directly with Qwen 3.5-4B and Gemma 4 E4B using the same data,
evaluation contexts and protocol:

| Step | What happens | Same in all notebooks |
|---|---|---|
| Data | `candidate_train` (1,000 SocraTeach + 1,000 MathDial), `candidate_val` (100 + 100) | yes |
| Evaluation set | 100 contexts (50 per source) and 100 validation dialogues (50 per source), fixed seed | yes |
| 1. Baseline | Pretrained model in 4-bit, no training | yes |
| 2. Tuning | Three LoRA settings, each trained on the same 400 candidate dialogues | yes |
| 3. Selection | Lowest validation loss among settings that do not leak more than real tutors (+2 points) | yes |
| 4. Final model | Best setting trained on all 2,000 candidate dialogues | yes |
| Generation | Greedy, stops at the end-of-turn token, max 96 new tokens | yes |

**How to run on Kaggle**

1. Settings: Accelerator **GPU T4 x2** (one GPU is used), Internet **on**.
2. Add Hariom's Kaggle dataset (https://www.kaggle.com/datasets/hariomkr/socratic-test-db) that holds `candidate_train.jsonl.gz` and `candidate_val.jsonl.gz`
   (1,000 + 1,000 and 100 + 100 dialogues). The notebook finds them anywhere under `/kaggle/input`.
3. Click **Save Version > Save & Run All (Commit)**. The run continues with the browser closed.
4. If a run stops early, add its output (the `phi4mini_socratic_final` folder) as an input dataset and
   commit again: finished models are skipped and an interrupted training resumes from its last checkpoint.

| Metric | Meaning | Better |
|---|---|---|
| leak rate | reply states the final answer before the student has said it | lower |
| question rate | reply contains a question | higher for a Socratic tutor |
| words | reply length | closer to real tutors |
| ROUGE-L, BLEU | word overlap with the real tutor's next turn | higher |
| val loss | masked loss on the target tutor turns (comparable within a model only, not across tokenizers) | lower |
| runaway replies | replies that ran on into an invented next turn and were cut | 0 |

In [ ]:
SMOKE_TEST = False     # True: tiny random model on CPU, a few examples (checks the code, not the results)

import os, socket, subprocess, sys

def pip_install(*packages, required=True):
    # Shows pip's own error instead of a bare exit status; retries for "externally managed" system Pythons
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode and "externally-managed" in r.stdout + r.stderr:
        r = subprocess.run(cmd + ["--break-system-packages"], capture_output=True, text=True)
    if r.returncode:
        print((r.stdout + r.stderr)[-4000:])
        if required:
            raise RuntimeError(f"pip install failed for {packages}; pip's message is printed above")
        print(f"optional install failed for {packages}; continuing without it")
    return r.returncode == 0

if not SMOKE_TEST:
    try:
        socket.create_connection(("pypi.org", 443), timeout=15).close()
    except OSError:
        raise RuntimeError("No internet access. Kaggle: Settings (right panel) > Internet > On "
                           "(needs a phone-verified account), then run again.")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle offers two T4s; this notebook uses one
if not SMOKE_TEST:
    pip_install("transformers>=5.0,<6", "peft", "accelerate", "bitsandbytes", "rouge_score", "sacrebleu")

In [ ]:
import gc, glob, gzip, json, random, re, shutil, time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from rouge_score import rouge_scorer
import sacrebleu
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Phi-4-mini"
MODEL_ID, REVISION = "microsoft/Phi-4-mini-instruct", "cfbefacb99257ffa30c83adab238a50856ac3083"

# ---- shared protocol: keep identical in the Qwen and Gemma notebooks ----
MAX_LEN = 768             # tokens per training dialogue (cut back to the last tutor turn that fits)
GRID_TRAIN = 400          # candidate dialogues used to compare LoRA settings (same dialogue IDs in both notebooks)
EVAL_PER_SOURCE = 50      # evaluation contexts per source
VAL_PER_SOURCE = 50       # validation dialogues per source for the loss
MAX_NEW_TOKENS = 96
LEAK_TOLERANCE = 0.02     # a setting may leak at most 2 points more than the real tutors on the same contexts
SAVE_STEPS = 25           # training checkpoint interval, for resuming after a disconnect
SEED = 42
GRID = [dict(r=8, lora_alpha=8, learning_rate=2e-4),
        dict(r=16, lora_alpha=16, learning_rate=2e-4),
        dict(r=8, lora_alpha=8, learning_rate=5e-5)]
COMMON = dict(lora_dropout=0.05, batch_size=1, grad_accum=8, warmup=0.05, epochs=1)
# ---- model-specific ----
TARGET_MODULES = ["qkv_proj", "o_proj", "gate_up_proj", "down_proj"]   # Phi-3 family fuses q/k/v and gate/up, so these cover the same attention and MLP projections
GEN_BATCH = 8

if SMOKE_TEST:
    MAX_LEN, GRID_TRAIN, EVAL_PER_SOURCE, VAL_PER_SOURCE, MAX_NEW_TOKENS, GEN_BATCH, SAVE_STEPS = 512, 6, 2, 2, 8, 2, 2
    GRID = GRID[:2]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = Path("/kaggle/working/phi4mini_socratic_final") if Path("/kaggle/working").exists() else Path("final_output_phi4mini")
(OUT / "figures").mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| output", OUT)

RUN_SETTINGS = dict(model=MODEL_ID, revision=REVISION, max_len=MAX_LEN, grid_train=GRID_TRAIN, eval_per_source=EVAL_PER_SOURCE,
                    val_per_source=VAL_PER_SOURCE, max_new_tokens=MAX_NEW_TOKENS, grid=GRID, common=COMMON, seed=SEED)
CACHE_FILE = OUT / "run_cache.json"   # finished models; a rerun skips them
if not CACHE_FILE.exists():            # Kaggle: reuse an earlier run's output attached as an input dataset
    for prev in sorted(glob.glob("/kaggle/input/**/run_cache.json", recursive=True)):
        if json.loads(Path(prev).read_text()).get("_settings") == json.loads(json.dumps(RUN_SETTINGS)):
            shutil.copytree(Path(prev).parent, OUT, dirs_exist_ok=True)
            print("resuming from", Path(prev).parent)
            break
CACHE = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
if CACHE.get("_settings") != json.loads(json.dumps(RUN_SETTINGS)):
    if CACHE:
        print("saved results used different settings; starting fresh")
    CACHE = {"_settings": RUN_SETTINGS}
def save_cache():
    CACHE_FILE.write_text(json.dumps(CACHE, indent=1, default=str))

RESULTS = {"config": dict(model_name=MODEL_NAME, **RUN_SETTINGS, smoke_test=SMOKE_TEST)}

## 1. Data

In [ ]:
def find_file(name):
    # .jsonl or .jsonl.gz on Kaggle, Colab (/content or Drive) or next to the notebook
    for pattern in [f"/kaggle/input/**/{name}.jsonl*", f"/content/{name}.jsonl*", f"/content/drive/MyDrive/**/{name}.jsonl*",
                    f"../socratic_data_prep/processed/{name}.jsonl*",
                    f"../../../hariom kumar/milestone-3/socratic_data_prep/processed/{name}.jsonl*", f"processed/{name}.jsonl*"]:
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(f"{name}.jsonl(.gz) not found")

def read_jsonl(path):
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows, val_rows = read_jsonl(find_file("candidate_train")), read_jsonl(find_file("candidate_val"))
counts = Counter(r["source"] for r in train_rows)
print("train", dict(counts), "| val", dict(Counter(r["source"] for r in val_rows)))
if not SMOKE_TEST:
    assert counts == {"socrateach": 1000, "mathdial": 1000}, "expected the new 1,000 + 1,000 candidate_train; re-upload it"

def per_source(rows, n, seed):
    rng = random.Random(seed)
    return [r for src in ["socrateach", "mathdial"] for r in rng.sample([x for x in rows if x["source"] == src],
                                                                        min(n, sum(x["source"] == src for x in rows)))]

eval_rows = per_source(val_rows, EVAL_PER_SOURCE, SEED)
loss_rows = per_source(val_rows, VAL_PER_SOURCE, SEED + 1)
# The tuning subset is chosen by dialogue ID before tokenisation, so both models tune on the same dialogues
grid_ids = set(random.Random(SEED).sample(sorted(r["dialogue_id"] for r in train_rows), min(GRID_TRAIN, len(train_rows))))
print("eval", dict(Counter(r["source"] for r in eval_rows)), "| val loss", dict(Counter(r["source"] for r in loss_rows)),
      "| tuning subset", dict(Counter(r["source"] for r in train_rows if r["dialogue_id"] in grid_ids)))

## 2. Prompt format and training targets

Each dialogue starts with a system prompt holding the tutoring instruction, the problem, the correct
final answer (for checking, never to reveal) and, for MathDial, the student's first attempt. Tutor
turns are the assistant role and student turns the user role. Tutor turns are training targets unless
MathDial labels them "telling" or they state the answer before the student has; those turns stay as
context without loss. The end-of-turn token is part of each target, so the model learns to stop.

The text comes from the model's own chat template (thinking off). Each tutor turn's content and
end-of-turn marker are located in that text, and the loss mask is set from their character offsets.

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
tok.pad_token = tok.pad_token or tok.eos_token

SYSTEM = ("You are a Socratic maths tutor. Guide the student one step at a time with questions and hints. "
          "Never state the final answer.\n\nProblem: {question}\n"
          "Correct final answer (for checking only, never reveal it): {answer}")

def chat_messages(rec, upto=None):
    sys_msg = SYSTEM.format(question=rec["question"], answer=rec["reference_answer"])
    if rec.get("student_attempt"):
        sys_msg += "\n\nStudent's first attempt:\n" + rec["student_attempt"]
    msgs = rec["messages"] if upto is None else rec["messages"][:upto]
    return [{"role": "system", "content": sys_msg}] + [
        {"role": "assistant" if m["role"] == "tutor" else "user", "content": m["content"]} for m in msgs]

def template(msgs, generation_prompt=False):
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=generation_prompt)

END_OF_TURN = "<|end|>"   # Phi-4-mini closes every turn with <|end|>; set by hand because the template adds <|endoftext|> after the last turn
_probe = template([{"role": "user", "content": "U"}, {"role": "assistant", "content": "QQQ"}])
assert "QQQ" + END_OF_TURN in _probe, "unexpected Phi-4-mini chat template"
eot_ids = tok.encode(END_OF_TURN, add_special_tokens=False)
STOP_IDS = sorted({tok.eos_token_id, *(eot_ids if len(eot_ids) == 1 else [])} - {None})
print("end-of-turn marker:", repr(END_OF_TURN), "| generation stops at token ids", STOP_IDS)

def render(rec, n_messages):
    msgs = chat_messages(rec, n_messages)
    if not any(m["role"] == "user" for m in msgs):   # a tutor opener with no student turn is no target
        return None, None
    text, spans, pos = template(msgs), [], 0
    for i, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        start = text.find(m["content"] + END_OF_TURN, pos)
        if start < 0:
            return None, None
        end = start + len(m["content"]) + len(END_OF_TURN)
        src = rec["messages"][i - 1]
        spans.append((start, end, src.get("move") != "telling" and not src.get("leaks_answer", False)))
        pos = end
    return text, spans

found = sum(render(r, len(r["messages"]))[0] is not None for r in train_rows[:200])
assert found == min(200, len(train_rows)), "tutor turns not found in the chat template text"

def encode(rec):
    n = len(rec["messages"])
    while n > 0:
        if rec["messages"][n - 1]["role"] == "tutor":
            text, spans = render(rec, n)
            if text is None:
                return None
            enc = tok(text, return_offsets_mapping=True, add_special_tokens=False)
            if len(enc["input_ids"]) <= MAX_LEN:
                break
        n -= 1
    if n == 0:
        return None
    labels = [-100] * len(enc["input_ids"])
    for k, (a, b) in enumerate(enc["offset_mapping"]):
        if any(train and a >= s and b <= e for s, e, train in spans):
            labels[k] = enc["input_ids"][k]
    if all(l == -100 for l in labels):
        return None
    return dict(dialogue_id=rec["dialogue_id"], input_ids=enc["input_ids"], labels=labels, cut=n < len(rec["messages"]))

train_all = [e for e in map(encode, train_rows) if e]
grid_enc = [e for e in train_all if e["dialogue_id"] in grid_ids]
val_enc = [e for e in map(encode, loss_rows) if e]
assert eot_ids[-1] in [l for e in train_all[:20] for l in e["labels"]], "end-of-turn token is not a training target"
RESULTS["data"] = dict(train_dialogues=len(train_all), dropped=len(train_rows) - len(train_all), tuning_dialogues=len(grid_enc),
                       val_dialogues=len(val_enc), train_tokens=sum(len(e["input_ids"]) for e in train_all),
                       median_tokens=int(np.median([len(e["input_ids"]) for e in train_all])),
                       shortened=sum(e["cut"] for e in train_all))
print(RESULTS["data"])

## 3. Evaluation contexts and metrics

For each evaluation dialogue, one tutor turn that follows a student message is chosen with a fixed
seed (the choice does not depend on the model). The model sees everything before it and writes the
next tutor message; the real tutor's message is the reference. Metrics are reported overall and per
source.

In [ ]:
NUM = re.compile(r"(?<![\d./])-?\d[\d,]*(?:\.\d+)?(?!\d|\.\d|/)")

def numbers(text):
    return {n.replace(",", "").rstrip("0").rstrip(".") if "." in n else n.replace(",", "") for n in NUM.findall(text)}

def answer_key(rec):
    found = numbers(rec["reference_answer"])
    key = found.pop() if len(found) == 1 else None
    return None if key in numbers(rec["question"]) else key

rng = random.Random(SEED)
eval_points = []
for rec in eval_rows:
    idx = [i for i, m in enumerate(rec["messages"]) if m["role"] == "tutor" and i > 0 and rec["messages"][i - 1]["role"] == "student"]
    if not idx:
        continue
    i = rng.choice(idx)
    key = answer_key(rec)
    said = key is not None and any(key in numbers(m["content"]) for m in rec["messages"][:i] if m["role"] == "student")
    eval_points.append(dict(dialogue_id=rec["dialogue_id"], turn=i, source=rec["source"], key=key, student_said=said,
                            prompt=template(chat_messages(rec, i), generation_prompt=True),
                            reference=rec["messages"][i]["content"]))
pd.DataFrame([{k: p[k] for k in ["dialogue_id", "turn", "source"]} for p in eval_points]).to_csv(OUT / "eval_contexts.csv", index=False)
print(len(eval_points), "evaluation contexts", dict(Counter(p["source"] for p in eval_points)))

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def score(replies, points):
    df = pd.DataFrame([dict(source=p["source"], leak=p["key"] is not None and not p["student_said"] and p["key"] in numbers(r),
                            question="?" in r, words=len(r.split()), rougeL=scorer.score(p["reference"], r)["rougeL"].fmeasure)
                       for r, p in zip(replies, points)])
    def summary(mask):
        sel = [r for r, m in zip(replies, mask) if m]
        refs = [p["reference"] for p, m in zip(points, mask) if m]
        d = df[mask]
        return {k: round(float(v), 4) for k, v in dict(leak_rate=d.leak.mean(), question_rate=d.question.mean(), words=d.words.mean(),
                                                       rougeL=d.rougeL.mean(), bleu=sacrebleu.corpus_bleu(sel, [refs]).score / 100).items()}
    out = summary(np.ones(len(df), bool))
    out["by_source"] = {src: summary((df.source == src).values) for src in ["socrateach", "mathdial"]}
    return out

RESULTS["real_tutors"] = score([p["reference"] for p in eval_points], eval_points)
print("real tutors:", {k: v for k, v in RESULTS["real_tutors"].items() if k != "by_source"})

## 4. Model helpers

In [ ]:
def load_model():
    if SMOKE_TEST:   # random 2-layer Phi-3 style model with the real vocabulary
        from transformers import Phi3Config, Phi3ForCausalLM
        cfg = Phi3Config(vocab_size=len(tok), hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                         num_attention_heads=4, num_key_value_heads=2, pad_token_id=tok.pad_token_id)
        return Phi3ForCausalLM(cfg).to(DEVICE)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                               bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float16, device_map={"": 0},
                                                 quantization_config=quant)
    with torch.no_grad():   # the T4 has no bf16: stop early if fp16 overflows instead of training on inf/nan
        e = val_enc[0]
        loss = model(input_ids=torch.tensor([e["input_ids"]], device=DEVICE), labels=torch.tensor([e["labels"]], device=DEVICE)).loss
    assert torch.isfinite(loss), "fp16 forward pass gave inf/nan on the T4"
    print("fp16 check loss:", round(float(loss), 3), "| GPU memory", round(torch.cuda.memory_allocated() / 2**30, 2), "GiB")
    return model

def free():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

RUNAWAY = re.compile(r"\n\s*(?:user|assistant|model|system)\s*\n", re.I)   # an invented next turn after a missed stop token

@torch.no_grad()
def generate(model, points):
    model.eval()
    tok.padding_side = "left"
    replies, runaway = [], 0
    cfg_eos = model.generation_config.eos_token_id
    stops = sorted(set(STOP_IDS) | set(cfg_eos if isinstance(cfg_eos, list) else [] if cfg_eos is None else [cfg_eos]))
    for i in range(0, len(points), GEN_BATCH):
        batch = tok([p["prompt"] for p in points[i:i + GEN_BATCH]], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        out = model.generate(**batch, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id, eos_token_id=stops)
        for seq in out[:, batch["input_ids"].shape[1]:]:
            text = re.sub(r"<think>.*?</think>", "", tok.decode(seq, skip_special_tokens=True), flags=re.S).strip()
            parts = RUNAWAY.split(text, maxsplit=1)
            runaway += len(parts) > 1
            replies.append(parts[0].strip())
    tok.padding_side = "right"
    return replies, runaway

@torch.no_grad()
def val_loss(model):
    model.eval()
    total, count = 0.0, 0
    for e in val_enc:
        ids, labels = torch.tensor([e["input_ids"]], device=DEVICE), torch.tensor([e["labels"]], device=DEVICE)
        n = int((labels[:, 1:] != -100).sum())
        total += float(model(input_ids=ids, labels=labels).loss) * n
        count += n
    return round(total / count, 4)

def evaluate(model, name):
    t = time.time()
    replies, runaway = generate(model, eval_points)
    metrics = {**score(replies, eval_points), "val_loss": val_loss(model), "runaway_replies": runaway,
               "empty_replies": sum(not r for r in replies), "eval_minutes": round((time.time() - t) / 60, 2)}
    pd.DataFrame(dict(dialogue_id=[p["dialogue_id"] for p in eval_points], source=[p["source"] for p in eval_points],
                      reference=[p["reference"] for p in eval_points], reply=replies)).to_csv(OUT / f"generations_{name}.csv", index=False)
    print(name, {k: v for k, v in metrics.items() if k != "by_source"})
    return metrics

## 5. Baseline: pretrained model in 4-bit, no training

In [ ]:
BASE = load_model()
if "baseline_4bit" not in CACHE:
    CACHE["baseline_4bit"] = evaluate(BASE, "baseline_4bit"); save_cache()
RESULTS["baseline_4bit"] = CACHE["baseline_4bit"]

## 6. LoRA training setup

Every run trains for one epoch with effective batch 8, cosine schedule, 5% warm-up, 8-bit paged AdamW,
gradient checkpointing and dropout 0.05. The 4-bit model is loaded once; each run gets a fresh adapter,
which is saved and then removed. Training checkpoints are written every `SAVE_STEPS` optimizer steps,
so a run interrupted by a disconnect resumes where it stopped.

In [ ]:
# Freeze the base and enable gradient checkpointing directly: prepare_model_for_kbit_training would cast all
# 16-bit weights to fp32, which does not fit on a T4 for Gemma's per-layer embeddings.
for p in BASE.parameters():
    p.requires_grad_(False)
missing = [t for t in TARGET_MODULES if not any(n.endswith(t) for n, _ in BASE.named_modules())]
assert not missing, f"LoRA target modules not found in the model: {missing}"
if DEVICE == "cuda":
    BASE.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
BASE.enable_input_require_grads()

def collate(batch):
    width = max(len(e["input_ids"]) for e in batch)
    pad = lambda seq, v: seq + [v] * (width - len(seq))
    return dict(input_ids=torch.tensor([pad(e["input_ids"], tok.pad_token_id) for e in batch]),
                labels=torch.tensor([pad(e["labels"], -100) for e in batch]),
                attention_mask=torch.tensor([pad([1] * len(e["input_ids"]), 0) for e in batch]))

class ListDataset(torch.utils.data.Dataset):
    def __init__(self, rows): self.rows = [dict(input_ids=e["input_ids"], labels=e["labels"]) for e in rows]
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def release_adapter(model):
    base = model.unload()
    if hasattr(base, "peft_config"):
        del base.peft_config
    assert not any("lora_" in n for n, _ in base.named_modules()), "LoRA layers left on the base model"
    return base

def run(name, setting, data):
    # Train one adapter, evaluate it, save it, and give the clean base model back
    global BASE
    if name in CACHE:
        print(name, "(already done)", {k: v for k, v in CACHE[name].items() if k not in ("history", "by_source")}); return
    torch.manual_seed(SEED)
    model = get_peft_model(BASE, LoraConfig(r=setting["r"], lora_alpha=setting["lora_alpha"], lora_dropout=COMMON["lora_dropout"],
                                            target_modules=TARGET_MODULES, bias="none", task_type="CAUSAL_LM"))
    ckpt_dir = OUT / "checkpoints" / name
    args = TrainingArguments(output_dir=str(ckpt_dir), num_train_epochs=COMMON["epochs"],
                             per_device_train_batch_size=COMMON["batch_size"], gradient_accumulation_steps=COMMON["grad_accum"],
                             learning_rate=setting["learning_rate"], lr_scheduler_type="cosine", warmup_steps=COMMON["warmup"],
                             logging_steps=1 if SMOKE_TEST else 5, save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=1,
                             report_to="none", seed=SEED, data_seed=SEED,
                             fp16=DEVICE == "cuda", optim="paged_adamw_8bit" if DEVICE == "cuda" else "adamw_torch",
                             gradient_checkpointing=DEVICE == "cuda", gradient_checkpointing_kwargs={"use_reentrant": False},
                             remove_unused_columns=False, max_steps=3 if SMOKE_TEST else -1)
    last = get_last_checkpoint(str(ckpt_dir)) if ckpt_dir.exists() else None
    if last:
        print(name, "resuming from", last)
    t = time.time()
    trainer = Trainer(model=model, args=args, train_dataset=ListDataset(data), data_collator=collate)
    trainer.train(resume_from_checkpoint=last)
    history = [dict(step=h["step"], loss=h["loss"]) for h in trainer.state.log_history if "loss" in h]
    model = trainer.model
    trainer.accelerator.free_memory(); del trainer; free()
    train_minutes = round((time.time() - t) / 60, 2)
    metrics = evaluate(model, name)
    model.save_pretrained(OUT / "adapters" / name)
    CACHE[name] = dict(**setting, **metrics, train_dialogues=len(data), train_minutes=train_minutes, history=history,
                       trainable_params=sum(p.numel() for p in model.parameters() if p.requires_grad))
    save_cache()
    shutil.rmtree(ckpt_dir, ignore_errors=True)   # finished: the saved adapter replaces the checkpoints
    BASE = release_adapter(model); del model; free()

## 7. Hyperparameter tuning on 400 candidate dialogues

Three settings: rank 8 at learning rate 2e-4, rank 16 at 2e-4, and rank 8 at 5e-5. Each trains on the
same 400 dialogues (about 50 optimizer steps) and is scored on the same contexts as the baseline.

In [ ]:
GRID_NAMES = [f"lora_r{s['r']}_lr{s['learning_rate']:g}" for s in GRID]
for name, setting in zip(GRID_NAMES, GRID):
    run(name, setting, grid_enc)

ok = [n for n in GRID_NAMES if CACHE[n]["leak_rate"] <= RESULTS["real_tutors"]["leak_rate"] + LEAK_TOLERANCE]
best = min(ok, key=lambda n: CACHE[n]["val_loss"]) if ok else min(GRID_NAMES, key=lambda n: (CACHE[n]["leak_rate"], CACHE[n]["val_loss"]))
RESULTS["grid"] = {n: {k: v for k, v in CACHE[n].items() if k != "history"} for n in GRID_NAMES}
RESULTS["best_setting"] = best
cols = ["leak_rate", "question_rate", "words", "rougeL", "bleu", "val_loss", "runaway_replies", "train_minutes"]
display(pd.DataFrame({n: CACHE[n] for n in GRID_NAMES}).T[cols])
print("selected:", best, "| rule: lowest val loss among settings with leak rate <=",
      round(RESULTS["real_tutors"]["leak_rate"] + LEAK_TOLERANCE, 3))

## 8. Final model: best setting on all 2,000 candidate dialogues

In [ ]:
FINAL = "final_" + best
setting = next(s for n, s in zip(GRID_NAMES, GRID) if n == best)
run(FINAL, setting, train_all)
RESULTS["final"] = {k: v for k, v in CACHE[FINAL].items() if k != "history"}
RESULTS["final_name"] = FINAL
print("final adapter saved to", OUT / "adapters" / FINAL)

## 9. Results

In [ ]:
rows = {"real tutors": RESULTS["real_tutors"], "baseline 4-bit": RESULTS["baseline_4bit"],
        **{n: CACHE[n] for n in GRID_NAMES}, FINAL: CACHE[FINAL]}
cols = ["leak_rate", "question_rate", "words", "rougeL", "bleu", "val_loss", "runaway_replies"]
overall = pd.DataFrame(rows).T.reindex(columns=cols)
print(MODEL_NAME, "- overall (100 contexts)"); display(overall)
per_src = pd.DataFrame({(name, src): m["by_source"][src] for name, m in rows.items() for src in ["socrateach", "mathdial"]}).T
print("per source (50 contexts each)"); display(per_src)

fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
for n in GRID_NAMES + [FINAL]:
    h = CACHE[n]["history"]
    ax[0].plot([x["step"] for x in h], [x["loss"] for x in h], label=n)
ax[0].set(title="Training loss", xlabel="step"); ax[0].legend(fontsize=7)
names = ["baseline 4-bit"] + GRID_NAMES + [FINAL]
ax[1].bar(names, [rows[n]["val_loss"] for n in names], color="#4C72B0")
ax[1].set(title="Validation loss (lower is better)"); ax[1].tick_params(axis="x", rotation=30)
x = np.arange(3)
show = ["real tutors", "baseline 4-bit", FINAL]
for k, n in enumerate(show):
    ax[2].bar(x + (k - 1) * 0.27, [rows[n]["leak_rate"], rows[n]["question_rate"], rows[n]["rougeL"]], 0.27, label=n)
ax[2].set_xticks(x, ["leak rate", "question rate", "ROUGE-L"]); ax[2].set(title="Reply metrics, same 100 contexts"); ax[2].legend(fontsize=7)
fig.suptitle(MODEL_NAME); fig.tight_layout(); fig.savefig(OUT / "figures/results.png"); plt.show()

side = pd.read_csv(OUT / "generations_baseline_4bit.csv").merge(
    pd.read_csv(OUT / f"generations_{FINAL}.csv")[["dialogue_id", "reply"]], on="dialogue_id", suffixes=("_baseline", "_final"))
display(side.groupby("source").head(3))

json.dump(RESULTS, open(OUT / "results.json", "w"), indent=1, default=str)
print("saved to", OUT, sorted(p.name for p in OUT.iterdir()))

## Outputs for the model comparison

`results.json` has the same structure in the Qwen and Gemma notebooks (`real_tutors`, `baseline_4bit`,
`grid`, `best_setting`, `final`, each with overall and `by_source` metrics), and `eval_contexts.csv`
lists the 100 contexts, which are the same dialogues and turns in both. The `generations_*.csv` files
hold every reply next to the real tutor's, for side-by-side reading.

Notes:
- Compare models on leak rate, question rate, words, ROUGE-L and BLEU. Validation loss depends on the tokenizer, so it is only comparable within one model.
- With 50 contexts per source, a difference of 1–2 leaks (2–4 points) is within noise.
- The leak check covers single-number final answers not already in the problem; question rate counts question marks.
- The MathDial test split (599 dialogues) is not used here; it stays for the final evaluation.